In [3]:
import os
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from rasterio.mask import mask as rst_mask
from datetime import datetime
from shapely.geometry import mapping
from rasterio.warp import calculate_default_transform, reproject, Resampling
from statsmodels.nonparametric.smoothers_lowess import lowess
from rasterio.windows import from_bounds

In [4]:
import os
from pathlib import Path

dir_path = Path(os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2019', 'NDVI', 'standardized_crs', 'NDVI_imputed'))
new_prefix = "2019"   # 👈 whatever you want to replace the first 4 letters with

for file in dir_path.iterdir():
    if file.is_file():
        old_name = file.name
        new_name = new_prefix + old_name[4:]   # replace first 4 letters
        file.rename(file.with_name(new_name))

# code for imputing all fields and special cases was not saved and need to be done again

In [5]:
def standardize_crs_and_clip_to_smallest_extent(raster_directory, target_crs, output_directory=None):
    """
    Reprojects all rasters in a directory to a target CRS (if needed) and clips them
    to the extent of the smallest raster (by pixel count) after reprojection.

    Parameters:
        raster_directory (str): Path to the directory containing raster files.
        target_crs (str): Target CRS in PROJ or EPSG format (e.g., "EPSG:32617").
        output_directory (str): Directory to save the output rasters. If None, creates a subfolder.

    Returns:
        None
    """
    from glob import glob

    if output_directory is None:
        output_directory = os.path.join(raster_directory, "standardized_crs_clipped")
    os.makedirs(output_directory, exist_ok=True)

    temp_dir = os.path.join(output_directory, "_temp_reprojected")
    os.makedirs(temp_dir, exist_ok=True)

    reprojected_paths = []
    smallest_area = float('inf')
    smallest_bounds = None
    smallest_transform = None

    # Step 1: Reproject rasters if needed, save to temp dir, and find smallest
    for path in glob(os.path.join(raster_directory, '*.tif')):
        filename = os.path.basename(path)
        output_path = os.path.join(temp_dir, filename)

        with rasterio.open(path) as src:
            if src.crs != target_crs:
                print(f"Reprojecting {filename} to {target_crs}")
                transform, width, height = calculate_default_transform(
                    src.crs, target_crs, src.width, src.height, *src.bounds
                )
                new_meta = src.meta.copy()
                new_meta.update({
                    'crs': target_crs,
                    'transform': transform,
                    'width': width,
                    'height': height
                })

                with rasterio.open(output_path, 'w', **new_meta) as dst:
                    for i in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, i),
                            destination=rasterio.band(dst, i),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=target_crs,
                            resampling=Resampling.nearest
                        )
            else:
                print(f"{filename} already matches {target_crs}. Copying without reprojection.")
                with rasterio.open(output_path, 'w', **src.meta) as dst:
                    dst.write(src.read())

        # Update the smallest raster info
        with rasterio.open(output_path) as reproj_src:
            pixel_count = reproj_src.width * reproj_src.height
            if pixel_count < smallest_area:
                smallest_area = pixel_count
                smallest_bounds = reproj_src.bounds
                smallest_transform = reproj_src.transform

        reprojected_paths.append(output_path)

    # Step 2: Clip all reprojected rasters to the smallest raster extent
    for path in reprojected_paths:
        filename = os.path.basename(path)
        final_output_path = os.path.join(output_directory, filename)

        with rasterio.open(path) as src:
            window = from_bounds(*smallest_bounds, transform=src.transform)
            clipped_data = src.read(window=window)
            clipped_transform = src.window_transform(window)

            clipped_meta = src.meta.copy()
            clipped_meta.update({
                'height': clipped_data.shape[1],
                'width': clipped_data.shape[2],
                'transform': clipped_transform
            })

            with rasterio.open(final_output_path, 'w', **clipped_meta) as dst:
                dst.write(clipped_data)

    # Cleanup temporary reprojection directory
    import shutil
    shutil.rmtree(temp_dir)

    print(f"All rasters have been reprojected (if needed) and clipped to smallest extent.\nSaved to: {output_directory}")

In [6]:
def standardize_crs(raster_directory, target_crs, output_directory=None):
    """
    Ensures all rasters in a directory have the same CRS. Rasters with a different CRS 
    are reprojected to the target CRS.

    Parameters:
        raster_directory (str): Path to the directory containing raster files.
        target_crs (str): Target CRS in PROJ or EPSG format (e.g., "EPSG:32617").
        output_directory (str): Directory to save the standardized rasters. 
                                If None, creates a subfolder in the input directory.

    Returns:
        None
    """
    if output_directory is None:
        output_directory = os.path.join(raster_directory, "standardized_crs")
    os.makedirs(output_directory, exist_ok=True)

    # Process each raster in the directory
    for filename in os.listdir(raster_directory):
        if filename.lower().endswith(('.tif', '.tiff')):
            file_path = os.path.join(raster_directory, filename)
            
            with rasterio.open(file_path) as src:
                # Check if the raster CRS matches the target CRS
                if src.crs != target_crs:
                    print(f"Reprojecting {filename} to {target_crs}")
                    
                    # Calculate transform and dimensions for the new CRS
                    transform, width, height = calculate_default_transform(
                        src.crs, target_crs, src.width, src.height, *src.bounds
                    )
                    
                    # Update metadata for the new CRS
                    new_meta = src.meta.copy()
                    new_meta.update({
                        "crs": target_crs,
                        "transform": transform,
                        "width": width,
                        "height": height
                    })
                    
                    # Output path for the reprojected raster
                    output_path = os.path.join(output_directory, filename)
                    
                    # Reproject and save the raster
                    with rasterio.open(output_path, "w", **new_meta) as dst:
                        for i in range(1, src.count + 1):  # Loop through raster bands
                            reproject(
                                source=rasterio.band(src, i),
                                destination=rasterio.band(dst, i),
                                src_transform=src.transform,
                                src_crs=src.crs,
                                dst_transform=transform,
                                dst_crs=target_crs,
                                resampling=Resampling.nearest
                            )
                else:
                    print(f"{filename} already matches {target_crs}. Copying to output directory.")
                    
                    # Copy the raster without reprojection
                    output_path = os.path.join(output_directory, filename)
                    new_meta = src.meta.copy()
                    with rasterio.open(output_path, "w", **new_meta) as dst:
                        dst.write(src.read())

    print(f"All rasters have been processed and saved to {output_directory}.")

In [7]:
def resize_to_largest(arrays):
    """
    Resizes a list of 2D arrays to the largest dimensions by padding with NaN.
    
    Parameters:
        arrays (list of np.ndarray): List of 2D arrays.
        
    Returns:
        np.ndarray: 3D stacked array.
    """
    # Find the largest dimensions
    max_rows = max(arr.shape[0] for arr in arrays)
    max_cols = max(arr.shape[1] for arr in arrays)
    
    # Create a list of padded arrays
    resized_arrays = []
    for arr in arrays:
        # Create an array of zeros with the target shape
        resized = np.zeros((max_rows, max_cols))
        resized[:arr.shape[0], :arr.shape[1]] = arr  # Place the original array into the padded array
        resized_arrays.append(resized)
    
    # Stack the arrays
    return np.stack(resized_arrays)

In [8]:
# Function to extract date from filename
def extract_date(filename):
    # Assuming filenames are in the format "YYYYMMDD_anything.tif"
    date_str = filename.split("_")[0]  # Modify this to match your filename format
    return datetime.strptime(date_str, "%Y%m%d")

In [9]:
def Impute_pixelwise_ndvi_lowess(PS_image_dir, field_boundary, target_crs):

    # Step 1: read PS_NDVI directory and make a new directory for imputed products
    PS_ndvi_dir = os.path.join(PS_image_dir, 'NDVI') # change the PS NDVI directory

    standardize_crs(PS_ndvi_dir, target_crs=target_crs, output_directory=None) # make sure all the rasters have the same crs
    PS_ndvi_dir = os.path.join(PS_ndvi_dir, 'standardized_crs')
    output_directory = os.path.join(PS_ndvi_dir, 'NDVI_imputed')
    os.makedirs(output_directory, exist_ok=True)

    # Step 2: Crop all rasters to the field boundary extent
    cropped_rasters = []
    dates = []
    transform = None
    crs = None

    for filename in os.listdir(PS_ndvi_dir):
        if filename.lower().endswith(('.tif', '.tiff')):
            date = extract_date(filename)
            file_path = os.path.join(PS_ndvi_dir, filename)
            with rasterio.open(file_path) as src:
                if crs is None:
                    crs = src.crs  # Store CRS of the first raster
                field_boundary_proj = field_boundary.to_crs(target_crs)
                cropped_image, cropped_transform = rst_mask(src, field_boundary_proj.buffer(0).geometry, crop=True) # buffer distance subjective
                cropped_rasters.append((date, cropped_image.squeeze()))
                dates.append(date)
                transform = cropped_transform

    # Ensure rasters and dates are sorted
    cropped_rasters.sort(key=lambda x: x[0])
    dates = sorted(dates)
    # Stack cropped rasters into a 3D array
    stack = np.array(resize_to_largest([x[1] for x in cropped_rasters]))  # Shape: (num_rasters, height, width)


    # Step 2: Generate a complete date range
    start_date = min(dates)
    end_date = max(dates)
    all_dates = pd.date_range(start=start_date, end=end_date)


    # Step 3: Apply LOWESS smoothing for each pixel
    imputed_stack = []
    date_indices = [(d - start_date).days for d in dates]  # Convert dates to numerical indices
    all_date_indices = [(d - start_date).days for d in all_dates]

    for i in range(stack.shape[1]):  # Iterate over rows
        imputed_row = []
        for j in range(stack.shape[2]):  # Iterate over columns
            # Extract the NDVI time series for this pixel
            time_series = stack[:, i, j]

            # Use LOWESS for smoothing
            smoothed = lowess(
                endog=time_series,  # NDVI values
                exog=date_indices,  # Corresponding time indices
                frac=0.3,           # Fraction of data used for smoothing
                it=0
            )

            # Interpolate missing values for all dates
            smoothed_values = pd.Series(smoothed[:, 1], index=smoothed[:, 0]).reindex(all_date_indices).interpolate()

            imputed_row.append(smoothed_values.values)

        # Rebuild the row stack
        imputed_stack.append(np.array(imputed_row).T)

    # Convert to 3D numpy array
    imputed_stack = np.array(imputed_stack).swapaxes(0, 1)


    # Step 4: Save imputed rasters
    for idx, date in enumerate(all_dates):
        output_path = os.path.join(output_directory, f"{date.strftime('%Y%m%d')}_PS_NDVI_imputed.tif")
        with rasterio.open(
            output_path,
            "w",
            driver="GTiff",
            height=imputed_stack[idx].shape[0],
            width=imputed_stack[idx].shape[1],
            count=1,
            dtype=imputed_stack[idx].dtype,
            crs=crs,
            transform=transform,
        ) as dst:
            dst.write(imputed_stack[idx], 1)
        print(f"Imputed raster saved: {output_path}")

In [10]:
# PA
# Gatesburg_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2019')
# Gatesburg_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2020')
# Gatesburg_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2021')
# Gatesburg_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2022')
# Gatesburg_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2023')
# Gatesburg_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2024')
#US_HWB_2017_PS_dir = os.path.join(os.getcwd(), 'Data', 'US-HWB')

# CA 
# Bi1_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2018')
# Bi1_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2019')
# Bi1_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2020')
# Bi1_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2021')
# Bi1_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2022')
# Bi1_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2023')
# Bi1_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2024')

# Bi2_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2018')
# Bi2_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2019')
# Bi2_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2020')
# Bi2_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2021')
# Bi2_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2022')
# Bi2_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2023')
# Bi2_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2024')

# Tw3_2017_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Tw3', '2017')
# Tw3_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Tw3', '2018')

# IL (yet to be downloaded and checked)
UiABC_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2018')
UiABC_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2019')
UiABC_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2020')
UiABC_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2021')
UiABC_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2022')
UiABC_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2023')
UiABC_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2024')

# IN (yet to be downloaded and checked)
VT12_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'IN', 'US-VT12', '2023')
VT12_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'IN', 'US-VT12', '2024')

In [12]:
# PA
# Impute_pixelwise_ndvi_lowess(Gatesburg_2023_PS_dir)

# CA
# Impute_pixelwise_ndvi_lowess(Bi1_2018_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi1_2019_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi1_2020_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi1_2021_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi1_2022_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi1_2023_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi1_2024_PS_dir, field_boundary, target_crs)

# Impute_pixelwise_ndvi_lowess(Bi2_2018_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi2_2019_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi2_2020_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi2_2021_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi2_2022_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi2_2023_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Bi2_2024_PS_dir, field_boundary, target_crs)

# Impute_pixelwise_ndvi_lowess(Tw3_2017_PS_dir, field_boundary, target_crs)
# Impute_pixelwise_ndvi_lowess(Tw3_2018_PS_dir, field_boundary, target_crs)

# IL
field_boundary = gpd.read_file(os.path.join('Boundaries', 'US-UiABC_Fields.geojson')).to_crs(epsg=4326)
target_crs = "EPSG:32616"
Impute_pixelwise_ndvi_lowess(UiABC_2018_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(UiABC_2019_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(UiABC_2020_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(UiABC_2021_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(UiABC_2022_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(UiABC_2023_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(UiABC_2024_PS_dir, field_boundary, target_crs)

# IN
field_boundary = gpd.read_file(os.path.join('Boundaries', 'US-VT12_Fields.geojson')).to_crs(epsg=4326)
target_crs = "EPSG:32616"
Impute_pixelwise_ndvi_lowess(VT12_2023_PS_dir, field_boundary, target_crs)
Impute_pixelwise_ndvi_lowess(VT12_2024_PS_dir, field_boundary, target_crs)

20180516_160647_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180524_160743_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180525_161027_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180526_160953_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180528_160907_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180601_160918_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180604_160901_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180605_160835_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180606_160853_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180616_160827_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180617_160931_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180706_160929_NDVI.tif already matches EPSG:32616. Copying to output directory.
20180707_160728_